In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers
import json
import os
import time
from openai import OpenAI


In [3]:
df = pd.read_csv('/content/synthetic_survey_data.csv')
df.head()

,Q1,Q2A,Q2B,Q2C,Q2D,Q2E,Q2F,Q2G,Q4A,Q4B,...,Q12I,Q13,Q14,Q15,Q16,Q17A,Q17B,Q17C,Q17D,Q17E
0,5.0,2.0,2.0,2.0,2.0,1.0,3.0,1.0,2.0,3.0,...,2.0,1.0,11.0,4.0,1.0,1.0,0.0,1.0,0.0,0.0
1,4.0,2.0,2.0,2.0,1.0,3.0,3.0,3.0,2.0,3.0,...,4.0,2.0,8.0,3.0,3.0,1.0,0.0,0.0,1.0,0.0
2,3.0,1.0,1.0,2.0,2.0,3.0,3.0,1.0,3.0,2.0,...,3.0,2.0,8.0,15.0,2.0,1.0,0.0,1.0,0.0,1.0
3,5.0,3.0,2.0,1.0,1.0,1.0,1.0,1.0,2.0,3.0,...,2.0,1.0,11.0,5.0,1.0,1.0,0.0,1.0,1.0,1.0
4,2.0,2.0,2.0,2.0,2.0,3.0,2.0,3.0,1.0,2.0,...,3.0,2.0,12.0,1.0,2.0,0.0,1.0,1.0,0.0,0.0


In [86]:
questions = pd.read_csv('/content/fragebogen.csv')
questions

,variable,question,options
0,Q0,Trinken Sie alkoholhaltiges oder alkoholfreies...,"['Ja(1)', 'Nein(2)']"
1,Q1,Wie häufig trinken Sie grundsätzlich alkoholfr...,"['täglich(1)', 'mehrmals pro Woche(2)', 'mehrm..."
2,Q2A,in Gesellschaft bei Freunden/Familie,"['nie(1)', 'gelegentlich(2)', 'häufig(3)', 'ke..."
3,Q2B,auf privaten Partys oder Feiern,"['nie(1)', 'gelegentlich(2)', 'häufig(3)', 'ke..."
4,Q2C,zuhause beim Essen,"['nie(1)', 'gelegentlich(2)', 'häufig(3)', 'ke..."
...,...,...,...
80,Q17A,alleinlebend,"['0', '1', 'keine Angabe(-99)']"
81,Q17B,mit Kind/Kindern lebend,"['0', '1', 'keine Angabe(-99)']"
82,Q17C,mit Eltern/Elternteil lebend,"['0', '1', 'keine Angabe(-99)']"
83,Q17D,mit Freund/in bzw. Lebenspartner/in lebend,"['0', '1', 'keine Angabe(-99)']"


In [87]:
selected_questions = questions[["variable", "question"]]
selected_questions.to_csv("fragebogen_kurz.csv", index=False)
selected_questions

,variable,question
0,Q0,Trinken Sie alkoholhaltiges oder alkoholfreies...
1,Q1,Wie häufig trinken Sie grundsätzlich alkoholfr...
2,Q2A,in Gesellschaft bei Freunden/Familie
3,Q2B,auf privaten Partys oder Feiern
4,Q2C,zuhause beim Essen
...,...,...
80,Q17A,alleinlebend
81,Q17B,mit Kind/Kindern lebend
82,Q17C,mit Eltern/Elternteil lebend
83,Q17D,mit Freund/in bzw. Lebenspartner/in lebend


# Question Scoring

In [134]:
# 1. OPEN AI Client erzeugen
client = OpenAI(api_key="")

In [135]:
# 2. Fragebogen-Datei hochladen
fragebogen_file = client.files.create(
    file=open("fragebogen_kurz.csv", "rb"),
    purpose="assistants"
)

In [136]:
# 3. Assistant für Genuss-Score-Bewertung erstellen
genuss_assistant = client.beta.assistants.create(
    name="GenussScoreEvaluator",
    instructions=(
        "Du bist ein Experte für sensorische Konsumforschung und Bierproduktentwicklung.\n"
        "Bewerte jede Umfragefrage im Hinblick auf ihre Relevanz für die Entwicklung eines genussorientierten Sensorik-Profils einer neuen Biersorte.\n"
        "Vergebe jeder Frage einen Genuss-Score (von 1–10), wobei:\n"
        "- 10: Frage ist hochrelevant für Geschmack, Aromawahrnehmung, Trinkfreude. Diese Fragen zielen auf die Geschmacksrichtung ab (z.B. süß, fruchtig oder Marken)\n"
        "- 5 Frage ist mittelbar relevant (z. B. Konsumkontext)\n"
        "- 1: Frage ist nicht relevant (z. B. rein soziodemografisch)\n\n"
        "Nutze dafür die angehängte Datei und suche nach einer Spalte, die die Fragen enthält. Generiere für jede Frage eine Spalte 'Genuss-Score'.\n"
        "Nach der Generierung des Scores generierst Du eine neue CSV-Datei namens 'scoring.csv'. Diese gibst Du dann zum Download zurück."
        "Stelle sicher, dass die Datei 'scoring.csv' über die Dateifunktion des Code Interpreters generiert und referenziert wird, damit sie heruntergeladen werden kann."
    ),
    model="gpt-4o",
    tools=[{"type": "code_interpreter"}],
    tool_resources={
        "code_interpreter": {
            "file_ids": [fragebogen_file.id]
        }
    }
)

# 4. Thread erstellen
genuss_thread = client.beta.threads.create()


/tmp/ipython-input-136-3130241769.py:25: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  genuss_thread = client.beta.threads.create()


In [137]:
# 5. Run starten
genuss_run = client.beta.threads.runs.create(
    thread_id=genuss_thread.id,
    assistant_id=genuss_assistant.id,
    instructions=(
        "Analysiere die angehängte Datei (CSV) und bewerte jede Frage wie in der Anweisung beschrieben (Scoring von 1-10).\n"
        "Dann generiere eine CSV-Datei namens 'scoring.csv' daraus. Diese gibst Du mir zum Download zurück."
        "Stelle sicher, dass die Datei 'scoring.csv' über die Dateifunktion des Code Interpreters generiert und referenziert wird, damit sie heruntergeladen werden kann."
    )
)

/tmp/ipython-input-137-2848655396.py:2: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  genuss_run = client.beta.threads.runs.create(


In [138]:
# 6. Auf Abschluss warten
while True:
    run_status = client.beta.threads.runs.retrieve(thread_id=genuss_thread.id, run_id=genuss_run.id)
    if run_status.status == "completed":
        break
    elif run_status.status == "failed":
        raise RuntimeError("Run für Genuss-Scoring fehlgeschlagen.")
    time.sleep(2)

/tmp/ipython-input-138-785899480.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run_status = client.beta.threads.runs.retrieve(thread_id=genuss_thread.id, run_id=genuss_run.id)


In [139]:
# 7. Nachrichten abrufen
genuss_messages = client.beta.threads.messages.list(thread_id=genuss_thread.id)

# 8. Suche nach CSV-Datei und lade sie herunter
file_id = None

for message in genuss_messages.data:
    for content in message.content:
        # (1) klassische textbasierte file_path Annotation
        if content.type == "text":
            annotations = getattr(content.text, "annotations", [])
            for annotation in annotations:
                if annotation.type == "file_path" and "scoring.csv" in annotation.text:
                    file_id = annotation.file_path.file_id

        # (2) direktes Dateielement (falls Datei als Ergebnisobjekt zurückkam)
        elif content.type == "file":
            file_id = content.file_id

        # (3) eingebettet in tool_calls (z. B. Code Interpreter)
        elif content.type == "tool_calls":
            for tool_call in content.tool_calls:
                if tool_call.type == "code":
                    for output in getattr(tool_call.code, "outputs", []):
                        if getattr(output, "type", None) == "file":
                            file_id = output.file_id

if not file_id:
    raise ValueError("CSV-Datei scoring.csv konnte nicht identifiziert werden.")


# 9. Datei herunterladen und als DataFrame laden
response = client.files.content(file_id)
file_bytes = response.read()


/tmp/ipython-input-139-1876545202.py:2: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  genuss_messages = client.beta.threads.messages.list(thread_id=genuss_thread.id)


In [142]:
with open("genuss_score.csv", "wb") as f:
    f.write(file_bytes)

# 10. Anzeige der Tabelle
score_df = pd.read_csv("genuss_score.csv")
score_df


,variable,question,score
0,Q0,Trinken Sie alkoholhaltiges oder alkoholfreies...,6
1,Q1,Wie häufig trinken Sie grundsätzlich alkoholfr...,1
2,Q2A,in Gesellschaft bei Freunden/Familie,4
3,Q2B,auf privaten Partys oder Feiern,4
4,Q2C,zuhause beim Essen,8
...,...,...,...
80,Q17A,alleinlebend,5
81,Q17B,mit Kind/Kindern lebend,5
82,Q17C,mit Eltern/Elternteil lebend,9
83,Q17D,mit Freund/in bzw. Lebenspartner/in lebend,5


In [144]:
score_df = pd.read_csv("genuss_score.csv")
score_df_filtered = score_df[score_df['score'] >= 8].copy()
score_df_filtered

,variable,question,score
4,Q2C,zuhause beim Essen,8.0
5,Q2D,zuhause zur Entspannung,10.0
21,Q6C,Es macht mir großen Spaß ein Bier oder bierhal...,10.0
25,Q7A,Gesunde Ernährung ist mir sehr wichtig.,8.0
27,Q7C,Ich probiere sehr gerne neue Produkte oder Ges...,9.0
31,Q8A,Geschmack des Getränks,8.0
32,Q8B,Geruch/Aroma des Getränks,9.0
33,Q8C,Kaufpreis des Getränks,8.0
35,Q8E,Nachhaltige Produktion des Getränks,8.0
37,Q8G,Design der Verpackung,8.0


In [146]:
score_df_filtered.to_csv("questions_genuss_score.csv", index=False)